In [366]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.sparse.linalg import cg

In [343]:
dt = 0.1
h = 0.1
rho = 1

In [344]:
N = 10
brzina_x = np.random.uniform(-2.0, 2.0, size=(N+1, N))
brzina_y = np.zeros((N, N+1))
pritisak = np.zeros((N,N))


In [345]:
import numpy as np

def Advekcija_brzina(brzina_x, brzina_y, dt = 0.1, h = 0.1):
    x_privremena = np.zeros_like(brzina_x)
    y_privremena = np.zeros_like(brzina_y)

    x_redovi, x_kolone = brzina_x.shape
    y_redovi, y_kolone = brzina_y.shape

    # --- PETLJA 1: Advekcija za horizontalnu brzinu (X) ---
    for i in range(x_redovi):
        for j in range(x_kolone):
            x_trenutno = brzina_x[i, j]

            # Pronađemo 4 komšije iz Y matrice da izračunamo prosek vertikalne brzine
            i_top = max(0, i - 1)
            i_bot = min(y_redovi - 1, i)
            j_left = max(0, j - 1)
            j_right = min(y_kolone - 1, j)

            # ISPRAVLJENO: Sva 4 različita ćoška su uključena
            y_prosek = (brzina_y[i_top, j_left] + brzina_y[i_bot, j_left] + \
                        brzina_y[i_top, j_right] + brzina_y[i_bot, j_right]) / 4

            # Skok unazad kroz vreme
            i_unazad = i - (y_prosek * dt / h)
            j_unazad = j - (x_trenutno * dt / h)

            # Ograničavanje unutar stola
            i_staro = max(0.0, min(float(x_redovi - 1), i_unazad))
            j_staro = max(0.0, min(float(x_kolone - 1), j_unazad))

            # Celobrojni ćoškovi
            i0 = int(np.floor(i_staro))
            i1 = min(x_redovi - 1, i0 + 1)
            j0 = int(np.floor(j_staro))
            j1 = min(x_kolone - 1, j0 + 1)

            # Težine (Udaljenosti)
            beta = i_staro - i0   # Vertikalni pomak (Y pravac)
            alfa = j_staro - j0   # Horizontalni pomak (X pravac)

            # ISPRAVLJENO: Čitamo iz brzina_x, uparene težine
            x_krajnje = brzina_x[i0, j0] * (1 - alfa) * (1 - beta) + \
                        brzina_x[i0, j1] * alfa * (1 - beta) + \
                        brzina_x[i1, j0] * (1 - alfa) * beta + \
                        brzina_x[i1, j1] * alfa * beta

            x_privremena[i, j] = x_krajnje
    
    # --- PETLJA 2: Advekcija za vertikalnu brzinu (Y) ---
    for i in range(y_redovi):
        for j in range(y_kolone):
            y_trenutno = brzina_y[i, j]

            # Pronađemo 4 komšije iz X matrice da izračunamo prosek horizontalne brzine
            i_top = max(0, i - 1)
            i_bot = min(x_redovi - 1, i)
            j_left = max(0, j - 1)
            j_right = min(x_kolone - 1, j)

            # ISPRAVLJENO: Sva 4 različita ćoška su uključena
            x_prosek = (brzina_x[i_top, j_left] + brzina_x[i_bot, j_left] + \
                        brzina_x[i_top, j_right] + brzina_x[x_redovi - 1 if i_bot >= x_redovi else i_bot, j_right]) / 4 
            # (Napomena: gornji indeks je obezbeđen da ne ispadne ako x_redovi ima drugačiju dimenziju)
            x_prosek = (brzina_x[i_top, j_left] + brzina_x[i_bot, j_left] + \
                        brzina_x[i_top, j_right] + brzina_x[min(x_redovi-1, i_bot), j_right]) / 4

            # Skok unazad kroz vreme
            i_unazad = i - (y_trenutno * dt / h)
            j_unazad = j - (x_prosek * dt / h)

            # ISPRAVLJENO: Ograničavamo i_unazad i j_unazad, a ne i_staro
            i_staro = max(0.0, min(float(y_redovi - 1), i_unazad))
            j_staro = max(0.0, min(float(y_kolone - 1), j_unazad))

            i0 = int(np.floor(i_staro))
            i1 = min(y_redovi - 1, i0 + 1)
            j0 = int(np.floor(j_staro))
            j1 = min(y_kolone - 1, j0 + 1)

            # ISPRAVLJENO: Ponovo računamo težine za trenutnu tačku
            beta = i_staro - i0
            alfa = j_staro - j0

            # ISPRAVLJENO: Čitamo iz brzina_y
            y_krajnje = brzina_y[i0, j0] * (1 - alfa) * (1 - beta) + \
                        brzina_y[i0, j1] * alfa * (1 - beta) + \
                        brzina_y[i1, j0] * (1 - alfa) * beta + \
                        brzina_y[i1, j1] * alfa * beta
            
            y_privremena[i, j] = y_krajnje

    # ISPRAVLJENO: Vraćamo x pa y, da se poklapa sa redosledom u definiciji
    return x_privremena, y_privremena

In [346]:
def Univerzalna_Advekcija(polje, brzina_x, brzina_y, tip_pozicije='centar', dt=0.1, h=0.1):
    novo_polje = np.zeros_like(polje)
    redova, kolone = polje.shape

    for i in range(redova):
        for j in range(kolone):
            
            if tip_pozicije == 'centar':
                
                u_polje = (brzina_x[i, j] + brzina_x[i, j + 1]) / 2.0
                v_polje = (brzina_y[i, j] + brzina_y[i + 1, j]) / 2.0
                
            elif tip_pozicije == 'x_ivica':
     
                u_polje = polje[i, j] 
                
                i_top = max(0, i - 1)
                i_bot = min(brzina_y.shape[0] - 1, i)
                j_left = max(0, j - 1)
                j_right = min(brzina_y.shape[1] - 1, j)
                v_polje = (brzina_y[i_top, j_left] + brzina_y[i_bot, j_left] + \
                           brzina_y[i_top, j_right] + brzina_y[i_bot, j_right]) / 4.0
                
            elif tip_pozicije == 'y_ivica':
                v_polje = polje[i, j] 
                
                i_top = max(0, i - 1)
                i_bot = min(brzina_x.shape[0] - 1, i)
                j_left = max(0, j - 1)
                j_right = min(brzina_x.shape[1] - 1, j)
                u_polje = (brzina_x[i_top, j_left] + brzina_x[i_bot, j_left] + \
                           brzina_x[i_top, j_right] + brzina_x[i_bot, j_right]) / 4.0

            i_unazad = i - (v_polje * dt / h)
            j_unazad = j - (u_polje * dt / h)

            i_staro = max(0.0, min(float(redova - 1), i_unazad))
            j_staro = max(0.0, min(float(kolone - 1), j_unazad))
         
            i0 = int(np.floor(i_staro))
            i1 = min(redova - 1, i0 + 1)
            j0 = int(np.floor(j_staro))
            j1 = min(kolone - 1, j0 + 1)

            beta = i_staro - i0
            alfa = j_staro - j0

            novo_polje[i, j] = polje[i0, j0] * (1 - alfa) * (1 - beta) + \
                               polje[i0, j1] * alfa * (1 - beta) + \
                               polje[i1, j0] * (1 - alfa) * beta + \
                               polje[i1, j1] * alfa * beta

    return novo_polje

In [347]:
def Viskoznost(brzina_x, brzina_y, dt, h, ni):
    x_privremena = np.zeros_like(brzina_x)
    y_privremena = np.zeros_like(brzina_y)

    x_redovi, x_kolone = brzina_x.shape
    y_redovi, y_kolone = brzina_y.shape

    for i in range(1, x_redovi - 1):
        for j in range(1, x_kolone - 1):
            viskoznost_x = ni * dt * (brzina_x[i - 1, j] + brzina_x[i, j - 1] + brzina_x[i, j + 1] + \
                                      brzina_x[i + 1, j + 1] - 4*brzina_x[i, j]) / (h**2) 
            x_privremena[i, j] = brzina_x[i, j] + viskoznost_x

    for i in range(1, y_redovi - 1):
        for j in range(1, y_kolone - 1):
            viskoznost_y = ni * dt * (brzina_y[i - 1, j] + brzina_y[i, j - 1] + brzina_y[i, j + 1] + \
                                      brzina_y[i + 1, j + 1] - 4*brzina_y[i, j]) / (h**2) 
            y_privremena[i, j] = brzina_y[i, j] + viskoznost_y
    
    return x_privremena, y_privremena

In [348]:
matrica = np.zeros((10,10))


In [349]:
tip_celije = [[0 , 0 , 0 , 0 , 0 , 0],
              [0 , 0 , 0 , 1 , 2 , 0],
              [0 , 2 , 2 , 1 , 0 , 0],
              [0 , 0 , 2 , 2 , 1 , 0],
              [0 , 1 , 0 , 1 , 0 , 0],
              [0 , 0 , 0 , 0 , 0 , 0]]
def IndexMap(tip_celije):
    
    N = len(tip_celije)

    mapa_indexa = np.zeros((N,N))
    cnt = 0
    for i in range(N):
        for j in range(N):
            if(tip_celije[i,j] == 0 or tip_celije[i,j] == 2):
                mapa_indexa[i,j] = -1
            else:
                mapa_indexa[i,j] = cnt
                cnt += 1
    
    return mapa_indexa
                

tip_celije_np = np.array(tip_celije)
mapa_indexa = IndexMap(tip_celije_np)


In [350]:
def Matrica_A(tip_celije):
    tip_celije = np.array(tip_celije)
    mapa_indexa = IndexMap(tip_celije)

    N = int(np.max(mapa_indexa) + 1)
    matrica_a = np.zeros((N, N)) 

    
    broj_redova, broj_kolona = mapa_indexa.shape

    
    for i in range(broj_redova):
        for j in range(broj_kolona):
            a = int(mapa_indexa[i, j])
            
            if a != -1:
                cnt = 0 
                
            
                if tip_celije[i-1, j] == 1:
                    cnt += 1
                    matrica_a[a, int(mapa_indexa[i-1, j])] = 1 
                if tip_celije[i+1, j] == 1:
                    cnt += 1
                    matrica_a[a, int(mapa_indexa[i+1, j])] = 1 
                if tip_celije[i, j-1] == 1:
                    cnt += 1
                    matrica_a[a, int(mapa_indexa[i, j-1])] = 1 
                if tip_celije[i, j+1] == 1:
                    cnt += 1
                    matrica_a[a, int(mapa_indexa[i, j+1])] = 1 

                
                if tip_celije[i-1, j] == 2:
                    cnt += 1
                if tip_celije[i+1, j] == 2:
                    cnt += 1
                if tip_celije[i, j-1] == 2:
                    cnt += 1
                if tip_celije[i, j+1] == 2:
                    cnt += 1

                matrica_a[a, a] = -cnt
                
    return matrica_a

In [351]:
matrica_a = Matrica_A(tip_celije)

In [352]:
matrica_a

array([[-2.,  1.,  0.,  0.,  0.],
       [ 1., -3.,  0.,  0.,  0.],
       [ 0.,  0., -1.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0., -1.]])

In [361]:
def vectorB(tip_celije, brzina_x, brzina_y, rho, dt):
    tip_celije = np.array(tip_celije)
    mapa_indexa = IndexMap(tip_celije)
    
    N = int(np.max(mapa_indexa) + 1)
    
    b = np.zeros(N)

    _, n = mapa_indexa.shape

    for i in range(n):
        for j in range(n):

            if (mapa_indexa[i,j] != -1) :

                u_levo = brzina_x[i, j]
                u_desno =brzina_x[i, j+1]
                v_gore = brzina_y[i,j]
                v_dole = brzina_y[i+1,j]

                if(tip_celije[i, j-1] == 0): u_levo = 0.0
                if(tip_celije[i, j+1] == 0): u_desno = 0.0
                if(tip_celije[i-1, j] == 0): v_gore = 0.0
                if(tip_celije[i+1, j] == 0): v_dole = 0.0

                divegencija = (u_desno - u_levo) + (v_dole - v_gore)

                b[int(mapa_indexa[i,j])] = -1 * (rho / dt) * divegencija

    return b

In [362]:
vektorB = vectorB(tip_celije, brzina_x, brzina_y, rho, dt)

In [363]:
vektorB

array([-17.57995766, -13.14950037,  13.14950037])

In [364]:
np.random.seed(42)

tip_celije = np.array([
    [0, 0, 0, 0],
    [0, 1, 2, 0],
    [0, 1, 1, 0],
    [0, 0, 0, 0]
])


u = np.random.uniform(-2.0, 2.0, size=(4, 5))
v = np.random.uniform(-2.0, 2.0, size=(5, 4))

rho = 1.0
dt = 0.1

# --- ISPIS ZA PROVERU ---
print("--- MATRICA TIP_CELIJE (4x4) ---")
print(tip_celije)
print("\n--- MATRICA BRZINA U (X brzina - 4x5) ---")
print(np.round(u, 2))
print("\n--- MATRICA BRZINA V (Y brzina - 5x4) ---")
print(np.round(v, 2))

vektorB = vectorB(tip_celije, u,v, rho, dt)
print("\n--- VEKTOR B   ---")
print(vektorB)

--- MATRICA TIP_CELIJE (4x4) ---
[[0 0 0 0]
 [0 1 2 0]
 [0 1 1 0]
 [0 0 0 0]]

--- MATRICA BRZINA U (X brzina - 4x5) ---
[[-0.5   1.8   0.93  0.39 -1.38]
 [-1.38 -1.77  1.46  0.4   0.83]
 [-1.92  1.88  1.33 -1.15 -1.27]
 [-1.27 -0.78  0.1  -0.27 -0.84]]

--- MATRICA BRZINA V (Y brzina - 5x4) ---
[[ 0.45 -1.44 -0.83 -0.53]
 [-0.18  1.14 -1.2   0.06]
 [ 0.37 -1.81  0.43 -1.32]
 [-1.74  1.8   1.86  1.23]
 [-0.78 -1.61  0.74 -0.24]]

--- VEKTOR B   ---
[  3.49493766 -31.43968912  17.59949971]


In [ ]:
def IzracunajPritisak(matrica_a, b_vektor, mapa_indexa, tip_celije, tol = 1e-5):
    P_vektor, _ = cg(matrica_a, b_vektor, tol=tol)

    P_matrica = np.zeros(tip_celije.shape)

    n = len(mapa_indexa)

    



_IncompleteInputError: incomplete input (3599082357.py, line 1)

5